In [ ]:
#!/usr/bin/env python3
"""
Calcolo dell'indice di attrattività comunale e generazione mappa.

Questo script:
1. Carica i dati OSM aggregati per comune
2. Calcola l'area di ogni comune dalla geometria
3. Normalizza i conteggi per superficie (densità per km²)
4. Applica normalizzazione min-max (0-100) per ogni pilastro
5. Calcola l'indice ponderato composito
6. Classifica i comuni in quintili (classe 1-5)
7. Genera una mappa coropletica interattiva in HTML

Librerire da installare: pip install pandas geopandas folium

Uso:
    python 02_calcolo_indice.ipynb

Output:
    - indicatore_di_attrattivita.csv: dati completi con indice e classe
    - mappa_attrattivita_sardegna.html: mappa interattiva in HTML
"""


In [ ]:
import pandas as pd
import geopandas as gpd
import folium 
from folium import MacroElement 
from branca.element import Element
from pathlib import Path
import numpy as np

In [62]:
# ============================================================
# CONFIGURAZIONE
# ============================================================

# Directory dei file di input/output
OUTPUT_DIR = "./output_attrattivita"

# File di input
GEOJSON_FILE = f"{OUTPUT_DIR}/comuni_sardegna.geojson"
CSV_FILE = f"{OUTPUT_DIR}/osm_per_comune.csv"

# File di output
INDICE_CSV = f"{OUTPUT_DIR}/indice_attrattivita.csv"
MAPPA_HTML = f"{OUTPUT_DIR}/mappa_attrattivita_sardegna.html"

In [64]:
# ============================================================
# CONFIGURAZIONE PESI
# ============================================================
# Pesi per il calcolo dell'indice composito.
# La somma deve essere 1.0 (o 100%).

WEIGHTS = {
    "turismo": 0.30,        # 30% - Turismo e patrimonio culturale
    "natura": 0.25,         # 25% - Natura e ambiente
    "ristorazione": 0.20,   # 20% - Ristorazione e commercio
    "servizi": 0.15,        # 15% - Servizi essenziali
    "infrastrutture": 0.10, # 10% - Infrastrutture di trasporto
}

In [65]:
# Pilastri da includere nel calcolo (devono corrispondere alle colonne del CSV)
PILLARS = list(WEIGHTS.keys())

In [66]:
# (chiesto all'ai) CRS proiettato per calcolo aree (UTM zone 32N per la Sardegna)
PROJECTED_CRS = "EPSG:32632"

# Coordinate centro mappa (Sardegna)
MAP_CENTER = [40.12, 9.01]
MAP_ZOOM = 8

In [67]:
# ============================================================
# FUNZIONI DI UTILITÀ
# ============================================================

def load_data(geojson_path: str, csv_path: str) -> tuple:
    """
    Carica i dati OSM e i confini comunali.
    
    Args:
        geojson_path: Percorso al file GeoJSON dei comuni
        csv_path: Percorso al file CSV con i conteggi OSM
        
    Returns:
        Tupla (GeoDataFrame comuni, DataFrame conteggi)
    """
    print("[1/7] Caricamento dati...")
    
    comuni_gdf = gpd.read_file(geojson_path)
    counts_df = pd.read_csv(csv_path)
    
    print(f"  → {len(comuni_gdf)} comuni caricati")
    print(f"  → {len(counts_df)} righe nel CSV")
    
    return comuni_gdf, counts_df


In [68]:
def calculate_areas(comuni_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Calcola l'area di ogni comune in km² usando un CRS proiettato.
    
    Args:
        comuni_gdf: GeoDataFrame con le geometrie comunali
        
    Returns:
        GeoDataFrame con nuova colonna 'area_km2'
    """
    print("[2/7] Calcolo aree comunali...")
    
    # Riproietta in CRS proiettato per calcolo aree accurate
    comuni_projected = comuni_gdf.to_crs(crs=PROJECTED_CRS)
    
    # Calcola area in km² (da m²)
    comuni_gdf = comuni_gdf.copy()
    comuni_gdf["area_km2"] = comuni_projected.geometry.area / 1_000_000
    
    # Calcola statistiche
    min_area = comuni_gdf["area_km2"].min()
    max_area = comuni_gdf["area_km2"].max()
    mean_area = comuni_gdf["area_km2"].mean()
    
    print(f"  → Area minima: {min_area:.2f} km²")
    print(f"  → Area massima: {max_area:.2f} km²")
    print(f"  → Area media: {mean_area:.2f} km²")
    
    return comuni_gdf

In [69]:
def normalize_by_area(counts_df: pd.DataFrame, comuni_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Normalizza i conteggi POI per superficie comunale (densità per km²).
    
    Args:
        counts_df: DataFrame con i conteggi POI per comune
        comuni_gdf: GeoDataFrame con le aree comunali
        
    Returns:
        DataFrame con nuove colonne di densità
    """
    print("[3/7] Normalizzazione per superficie...")
    
    # Merge per ottenere le aree
    result_df = counts_df.copy()
    
    # Il codice_istat deve essere un numero in entrambi
    comuni_areas = comuni_gdf[["com_istat_code", "area_km2"]].copy()
    comuni_areas["com_istat_code"] = comuni_areas["com_istat_code"].astype(int)
    
    result_df = result_df.merge(
        comuni_areas, 
        left_on="codice_istat", 
        right_on="com_istat_code", 
        how="left"
    )
    
    # Calcola densità per ogni pilastro
    for pillar in PILLARS:
        if pillar in result_df.columns:
            # Densità = count / area (valori piccoli, es. 0.5 servizi/km²)
            result_df[f"{pillar}_density"] = result_df[pillar] / result_df["area_km2"]
            result_df[f"{pillar}_log_density"] = np.log(result_df[pillar] / result_df["area_km2"] + 1)
        else:
            result_df[f"{pillar}_density"] = 0
            result_df[f"{pillar}_log_density"] = 0
    
    return result_df

In [ ]:
def min_max_normalize(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Applica normalizzazione min-max per scalare i valori in range 0-100.
    
    Argomenti:
        df: DataFrame con i valori da normalizzare
        columns: Lista delle colonne da normalizzare
        
    Risultato:
        DataFrame con nuove colonne normalizzate
    """
    result_df = df.copy()
    
    for col in columns:
        if col in result_df.columns:
            min_val = result_df[col].min()
            max_val = result_df[col].max()
                
            if max_val > min_val:
                # Min-max normalization: (x - min) / (max - min) * 100
                result_df[f"{col}_norm"] = (
                    (result_df[col] - min_val) / (max_val - min_val) * 100
                )
            else:
                result_df[f"{col}_norm"] = 0
            
            print(f"  → {col}: min={min_val:.4f}, max={max_val:.4f}")
        else:
            result_df[f"{col}_norm"] = 0
    
    return result_df

In [125]:
def calculate_weighted_index(df: pd.DataFrame, weights: dict) -> pd.DataFrame:
    """
    Calcola l'indice composito ponderato.
    
    Args:
        df: DataFrame con le colonne normalizzate
        weights: Dizionario {pilastro: peso} (somma = 1.0)
        
    Returns:
        DataFrame con nuova colonna 'indice_attrattivita_assoluto' e 'indice_attrattivita_log'
    """
    print("[4/7] Calcolo indice ponderato...")
    
    result_df = df.copy()
    
    # Calcola l'indice come somma ponderata dei pilastri normalizzati
    result_df["indice_attrattivita_assoluto"] = 0.0
    result_df["indice_attrattivita_og"] = 0.0
    
    for pillar, weight in weights.items():
        norm_col = f"{pillar}_density_norm"
        if (norm_col in result_df.columns):
            result_df["indice_attrattivita_assoluto"] += result_df[norm_col] * weight

    result_df["indice_attrattivita_log"] = np.log1p(result_df["indice_attrattivita_assoluto"])

    print(f"  → {pillar}: peso {weight*100:.0f}%")
    
    print(f"  → Indice range: {result_df['indice_attrattivita_assoluto'].min():.2f} - {result_df['indice_attrattivita_assoluto'].max():.2f}")
    print(f"  → Indice range log: {result_df['indice_attrattivita_log'].min():.2f} - {result_df['indice_attrattivita_log'].max():.2f}")
    
    return result_df

In [ ]:
def classify_quintiles(df: pd.DataFrame, column: str = "indice") -> pd.DataFrame:
    """
    Classifica i comuni in quintili (5 classi di attrattività).
    
    Args:
        df: DataFrame con l'indice
        column: Colonna da usare per la classificazione
        
    Returns:
        DataFrame con nuova colonna 'classe' (1-5)
    """
    print("[5/7] Classificazione in quintili...")
    
    result_df = df.copy()
    
    # Calcola i quintili (5 classi)
    # qcut assegna etichette 1-5 basate sui percentili
    result_df["classe"] = pd.qcut(
        result_df[column], 
        q=5, 
        labels=[1, 2, 3, 4, 5]
    ).astype(int)
    
    # Statistiche per classe
    for classe in range(1, 6):
        subset = result_df[result_df["classe"] == classe]
        print(f"  → Classe {classe}: {len(subset)} comuni "
              f"(indice {subset[column].min():.2f} - {subset[column].max():.2f})")
    
    return result_df

In [161]:
#non utilizzata
""" 
def save_results(df: pd.DataFrame, output_path: str) -> None:
    
    Salva il DataFrame risultato in CSV.
    
    Args:
        df: DataFrame con i risultati
        output_path: Percorso del file di output
    
    print(f"[6/7] Salvataggio risultati in {output_path}...")
    
    # Seleziona le colonne rilevanti per il CSV
    output_df = df[[
        "codice_istat", 
        "nome_comune", 
        "area_km2",
        "turismo", "natura", "ristorazione", "servizi", "infrastrutture",
        "indice", 
        "classe"
    ]].copy()
    
    # Arrotonda i valori numerici
    output_df["area_km2"] = output_df["area_km2"].round(2)
    output_df["indice"] = output_df["indice"].round(2)
    
    output_df.to_csv(output_path, index=False, encoding="utf-8")
    print(f"  → {len(output_df)} righe salvate")
"""

' \ndef save_results(df: pd.DataFrame, output_path: str) -> None:\n\n    Salva il DataFrame risultato in CSV.\n\n    Args:\n        df: DataFrame con i risultati\n        output_path: Percorso del file di output\n\n    print(f"[6/7] Salvataggio risultati in {output_path}...")\n\n    # Seleziona le colonne rilevanti per il CSV\n    output_df = df[[\n        "codice_istat", \n        "nome_comune", \n        "area_km2",\n        "turismo", "natura", "ristorazione", "servizi", "infrastrutture",\n        "indice", \n        "classe"\n    ]].copy()\n\n    # Arrotonda i valori numerici\n    output_df["area_km2"] = output_df["area_km2"].round(2)\n    output_df["indice"] = output_df["indice"].round(2)\n\n    output_df.to_csv(output_path, index=False, encoding="utf-8")\n    print(f"  → {len(output_df)} righe salvate")\n'

In [156]:
result_df.head()

,codice_istat,nome_comune,codice_istat_str,infrastrutture,natura,ristorazione,servizi,turismo,com_istat_code,area_km2,...,natura_density_norm,ristorazione_density_norm,servizi_density_norm,infrastrutture_density_norm,indice_attrattivita_assoluto,indice_attrattivita_og,indice_attrattivita_log,indice_attrattivita_assoluto_norm,indice_attrattivita_log_norm,classe
0,112001,Alghero,112001,238,301,263,68,298,112001,225.428280,...,9.456409,17.565300,8.543225,12.735734,13.247352,0.0,2.656571,16.410773,56.059048,5
1,112002,Anela,112002,1,190,1,3,6,112002,36.780588,...,36.585002,0.409345,2.310064,0.327972,10.117675,0.0,2.408536,12.345827,49.677422,5
2,112003,Ardara,112003,2,14,3,4,8,112003,38.172658,...,2.597430,1.183252,2.967761,0.632023,2.077163,0.0,1.124008,1.902494,16.628122,2
3,112004,Banari,112004,0,8,3,2,11,112004,21.361930,...,2.652269,2.114409,2.651617,0.000000,3.300881,0.0,1.458820,3.491908,25.242413,3
4,112005,Benetutti,112005,0,153,7,4,19,112005,94.066155,...,11.519303,1.120399,1.204337,0.000000,3.939153,0.0,1.597194,4.320921,28.802603,4


In [159]:
def generate_map(comuni_gdf: gpd.GeoDataFrame, result_df: pd.DataFrame, output_path: str) -> None:
    """
    Genera una mappa coropletica interattiva con folium.
    
    Args:
        comuni_gdf: GeoDataFrame con le geometrie comunali
        result_df: DataFrame con indice e classe
        output_path: Percorso del file HTML di output
    """
    print("[7/7] Generazione mappa coropletica...")
    
    # Prepara i dati per la mappa
    comuni_for_map = comuni_gdf.merge(
        result_df[["codice_istat", "indice_attrattivita_log_norm", "classe"] + 
                  [f"{p}_density_norm" for p in PILLARS]],
        left_on="com_istat_code_num",
        right_on="codice_istat",
        how="left"
    )
    
    # Crea la mappa base
    m = folium.Map(
        location=MAP_CENTER,
        zoom_start=MAP_ZOOM,
        tiles="CartoDB positron"
    )
    
    # Costruisci il GeoJSON inline per folium
    geojson_data = comuni_for_map.to_json()
    
    # Crea la mappa coropletica
    folium.Choropleth(
        geo_data=geojson_data,
        name="Attrattività",
        data=comuni_for_map,
        columns=["com_istat_code", "indice_attrattivita_log_norm"],
        key_on="feature.properties.com_istat_code",
        fill_color="YlOrRd",
        fill_opacity=0.7,
        line_opacity=0.2,
        line_weight=1,
        legend_name="Indice di Attrattività",
        nan_fill_color="white",
        nan_fill_opacity=0.3
    ).add_to(m)
    
    # Aggiungi tooltip interattivo
    style_function = lambda x: {
        "fillColor": "#ffffff",
        "color": "#000000",
        "fillOpacity": 0,
        "weight": 0
    }
    
    highlight_function = lambda x: {
        "fillColor": "#000000",
        "color": "#000000",
        "fillOpacity": 0.1,
        "weight": 1
    }
    
    # Tooltip con informazioni dettagliate
    tooltip_cols = [
        "name", "indice_attrattivita_log_norm", "classe",
        "turismo__density_norm", "natura__density_norm", "ristorazione__density_norm",
        "servizi__density_norm", "infrastrutture__density_norm"
    ]
    
    # Rinomina per il tooltip
    comuni_for_map["nome_comune"] = comuni_for_map["name"]
    
    # Crea il layer GeoJson con tooltip
    folium.GeoJson(
        geojson_data,
        name="Dettagli comune",
        style_function=style_function,
        highlight_function=highlight_function,
        tooltip=folium.GeoJsonTooltip(
            fields=["name", "indice_attrattivita_log_norm", "classe",
                    "turismo_density_norm", "natura_density_norm", "ristorazione_density_norm",
                    "servizi_density_norm", "infrastrutture_density_norm"],
            aliases=["Comune:", "Indice:", "Classe (1-5):",
                    "Turismo:", "Natura:", "Ristorazione:",
                    "Servizi:", "Infrastrutture:"],
            localize=True,
            sticky=True,
            labels=True,
            style="""
                background-color: white;
                border: 2px solid black;
                border-radius: 3px;
                box-shadow: 3px 3px 3px rgba(0,0,0,0.3);
                font-size: 12px;
                padding: 10px;
            """
        )
    ).add_to(m)
    
    # Aggiungi pannello info HTML custom (top-right)
    info_html = """
    <div style="position: fixed; 
                top: 10px; right: 10px; 
                width: 220px;
                background-color: white; 
                border: 2px solid grey; 
                border-radius: 5px;
                padding: 10px;
                z-index: 9999;
                font-size: 12px;
                box-shadow: 3px 3px 3px rgba(0,0,0,0.3);">
        <h4 style="margin: 0 0 10px 0; border-bottom: 1px solid grey; padding-bottom: 5px;">
            Indice di Attrattività
        </h4>
        <p style="margin: 5px 0;">
            <b>5 pilastri tematici</b><br>
            Turismo, Natura, Ristorazione,<br>
            Servizi, Infrastrutture
        </p>
        <p style="margin: 5px 0;">
            <b>Classi di attrattività:</b><br>
            1 = Molto bassa<br>
            2 = Bassa<br>
            3 = Media<br>
            4 = Alta<br>
            5 = Molto alta
        </p>
        <p style="margin: 5px 0; font-size: 10px; color: #666;">
            Passa il mouse sui comuni per i dettagli
        </p>
    </div>
    """
    m.get_root().html.add_child(Element(info_html))
    
    # Aggiungi legenda custom (bottom-left)
    legend_html = """
    <div style="position: fixed; 
                bottom: 50px; left: 10px; 
                width: 150px;
                background-color: white; 
                border: 2px solid grey; 
                border-radius: 5px;
                padding: 10px;
                z-index: 9999;
                font-size: 11px;
                box-shadow: 3px 3px 3px rgba(0,0,0,0.3);">
        <h4 style="margin: 0 0 10px 0;">Legenda</h4>
        <div style="background: linear-gradient(to right, #FFFFB2, #FED976, #FEB24C, #FD8D3C, #FC4E2A, #E31A1C, #B10026); 
                    height: 20px; 
                    border-radius: 3px;
                    margin-bottom: 5px;"></div>
        <div style="display: flex; justify-content: space-between; font-size: 10px;">
            <span>Basso</span>
            <span>Alto</span>
        </div>
    </div>
    """
    m.get_root().html.add_child(Element(legend_html))
    
    # Aggiungi controllo layer
    folium.LayerControl().add_to(m)
    
    # Salva la mappa
    m.save(output_path)
    print(f"  → Mappa salvata: {output_path}")

# ============================================================
# MAIN
# ============================================================


In [76]:
# Verifica che i file di input esistano
geojson_path = Path(GEOJSON_FILE)
csv_path = Path(CSV_FILE)

if not geojson_path.exists():
    print(f"ERRORE: File non trovato: {GEOJSON_FILE}")
    print("Esegui prima: python 01_raccolta_dati_osm.py")

if not csv_path.exists():
    print(f"ERRORE: File non trovato: {CSV_FILE}")
    print("Esegui prima: python 01_raccolta_dati_osm.py")

In [77]:
# 1. Carica i dati
comuni_gdf, counts_df = load_data(GEOJSON_FILE, CSV_FILE)

[1/7] Caricamento dati...
  → 377 comuni caricati
  → 377 righe nel CSV


In [78]:
# 2. Calcola le aree
comuni_gdf = calculate_areas(comuni_gdf)

[2/7] Calcolo aree comunali...
  → Area minima: 2.62 km²
  → Area massima: 547.32 km²
  → Area media: 63.95 km²


In [126]:
# 3. Normalizza per superficie
result_df = normalize_by_area(counts_df, comuni_gdf)

[3/7] Normalizzazione per superficie...


In [128]:
# 4. Applica min-max normalization per ogni pilastro
density_cols = [f"{p}_density" for p in PILLARS]
density_log_cols = [f"{p}_log_density" for p in PILLARS]
result_df = min_max_normalize(result_df, density_cols)


  → turismo_density: min=0.0258, max=8.1012
  → natura_density: min=0.0000, max=14.1199
  → ristorazione_density: min=0.0000, max=6.6419
  → servizi_density: min=0.0000, max=3.5308
  → infrastrutture_density: min=0.0000, max=8.2898


In [129]:
# 5. Calcola l'indice ponderato
result_df = calculate_weighted_index(result_df, WEIGHTS)

[4/7] Calcolo indice ponderato...
  → infrastrutture: peso 10%
  → Indice range: 0.61 - 77.60
  → Indice range log: 0.48 - 4.36


In [130]:
# 5.bis Calcola l'indice ponderato logaritmico normalizzato
indici_attrattivita_da_normalizzare = ["indice_attrattivita_assoluto", "indice_attrattivita_log"]
result_df = min_max_normalize(result_df, indici_attrattivita_da_normalizzare)

  → indice_attrattivita_assoluto: min=0.6124, max=77.6042
  → indice_attrattivita_log: min=0.4777, max=4.3644


In [132]:
# 6. Classifica in quintili
result_df = classify_quintiles(result_df, "indice_attrattivita_assoluto") 

[5/7] Classificazione in quintili...
  → Classe 1: 76 comuni (indice 0.61 - 1.67)
  → Classe 2: 75 comuni (indice 1.68 - 2.57)
  → Classe 3: 75 comuni (indice 2.59 - 3.88)
  → Classe 4: 75 comuni (indice 3.89 - 5.86)
  → Classe 5: 76 comuni (indice 5.87 - 77.60)


In [ ]:
# 7. Genera la mappa
generate_map(comuni_gdf, result_df, MAPPA_HTML)

[7/7] Generazione mappa coropletica...
  → Mappa salvata: ./output_attrattivita/mappa_attrattivita_sardegna.html


In [134]:
result_df.to_csv(f"{OUTPUT_DIR}/indicatore_di_attrattivita.csv")